# evaluation

In [1]:
import json
import time
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/opt/homebrew/anaconda3/envs/kg_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/homebrew/anaconda3/envs/kg_env/lib/python3.10/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/var/folders/r7/4ftfzdz96zs6t9xjbnmy5w3c0000gn/T/

In [11]:
from openai import OpenAI
from neo4j import GraphDatabase

# CONFIG 
MODEL_PROVIDER  = "openai"        
MODEL_NAME      = "gpt-4o-mini"   
OPENAI_API_KEY  = "your_openai_api_key_here"
GEMINI_API_KEY  = "your-gemini-key-here"  

NEO4J_URI       = "bolt://localhost:7687"
NEO4J_USER      = "neo4j"
NEO4J_PASSWORD  = "your_neo4j_password_here"
NEO4J_DATABASE  = "kg200city"
EMBEDDING_MODEL = "text-embedding-3-small"

client = OpenAI(api_key=OPENAI_API_KEY)
kg = Neo4jKG(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, "kg200city")
print("Connected!")

Connected!


In [12]:
langchain_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)
langchain_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

ragas_llm = LangchainLLMWrapper(langchain_llm)

/var/folders/r7/4ftfzdz96zs6t9xjbnmy5w3c0000gn/T/ipykernel_37368/2234416353.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(langchain_llm)


In [5]:
queries = [
    {"id": 1,  "type": "single_hop",  "question": "What climate hazards does Houston face?"},
    {"id": 2,  "type": "single_hop",  "question": "What adaptation actions has Vancouver implemented to address flooding?"},
    {"id": 3,  "type": "single_hop",  "question": "What policies has Calgary issued for climate resilience?"},
    {"id": 4,  "type": "single_hop",  "question": "What outcomes are produced by green stormwater infrastructure in Houston?"},
    {"id": 5,  "type": "single_hop",  "question": "Which actors are involved in climate governance in Washington DC?"},
    {"id": 6,  "type": "single_hop",  "question": "What adaptation actions address extreme heat in Pittsburgh?"},
    {"id": 7,  "type": "single_hop",  "question": "What vulnerabilities does Vancouver face related to sea level rise?"},
    {"id": 8,  "type": "single_hop",  "question": "What mechanisms does Calgary use to finance climate adaptation?"},
    {"id": 9,  "type": "single_hop",  "question": "Which cities have implemented nature-based solutions?"},
    {"id": 10, "type": "single_hop",  "question": "What adaptation actions address flooding in Washington DC?"},
    {"id": 11, "type": "multi_hop",   "question": "How do Houston's flood policies connect to specific measurable outcomes?"},
    {"id": 12, "type": "multi_hop",   "question": "What is the causal chain from extreme heat hazard to adaptation actions in Calgary?"},
    {"id": 13, "type": "multi_hop",   "question": "Which actors mandate adaptation actions that produce resilience outcomes in Vancouver?"},
    {"id": 14, "type": "multi_hop",   "question": "How does flooding vulnerability in Pittsburgh lead to specific infrastructure interventions?"},
    {"id": 15, "type": "multi_hop",   "question": "What is the pathway from climate hazard identification to policy issuance in Houston?"},
    {"id": 16, "type": "multi_hop",   "question": "How do green infrastructure actions in Vancouver address multiple hazard types?"},
    {"id": 17, "type": "multi_hop",   "question": "Which adaptation actions in Calgary produce outcomes related to both flooding and heat?"},
    {"id": 18, "type": "multi_hop",   "question": "How does Washington DC's governance structure connect hazards to adaptation outcomes?"},
    {"id": 19, "type": "multi_hop",   "question": "What is the relationship between Actor mandates and Outcome production in Pittsburgh?"},
    {"id": 20, "type": "multi_hop",   "question": "How do vulnerability assessments in Houston lead to specific policy mechanisms?"},
    {"id": 21, "type": "cross_city",  "question": "How do Vancouver and Calgary differ in their approaches to flood resilience?"},
    {"id": 22, "type": "cross_city",  "question": "Which cities have the most diverse portfolio of adaptation actions?"},
    {"id": 23, "type": "cross_city",  "question": "Compare the governance structures of Houston and Pittsburgh in climate adaptation."},
    {"id": 24, "type": "cross_city",  "question": "Which cities address both flooding and extreme heat simultaneously?"},
    {"id": 25, "type": "cross_city",  "question": "How do Houston and Vancouver differ in their climate policy mechanisms and outcomes?"},
]
print(f"Query set ready: {len(queries)} questions")

Query set ready: 25 questions


In [9]:
import json
import time
from tqdm import tqdm
from openai import OpenAI
from neo4j import GraphDatabase


# Neo4j connection

class Neo4jKG:
    def __init__(self, uri, user, password, database):
        self.driver   = GraphDatabase.driver(uri, auth=(user, password))
        self.database = database

    def run(self, query, params=None):
        with self.driver.session(database=self.database) as session:
            result = session.run(query, params or {})
            return [dict(r) for r in result]

    def run_write(self, query, params=None):
        with self.driver.session(database=self.database) as session:
            session.execute_write(lambda tx: tx.run(query, params or {}))

    def close(self):
        self.driver.close()


client = OpenAI(api_key=OPENAI_API_KEY)
kg     = Neo4jKG(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, "kg200city")



#Vector index creation

def embed(text: str) -> list:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding


def build_vector_index():
    print("Building vector embeddings for all node types...")

    # --- AdaptationAction ---
    actions = kg.run("""
        MATCH (a:AdaptationAction)
        WHERE a.embedding IS NULL
        RETURN elementId(a) AS eid,
               coalesce(a.action_name, '') + ' ' +
               coalesce(a.adaptation_type, '') + ' ' +
               coalesce(a.co_benefits, '') AS text
    """)
    for row in tqdm(actions, desc="Embedding AdaptationAction"):
        if not row['text'].strip():
            continue
        vec = embed(row['text'])
        kg.run_write("""
            MATCH (a:AdaptationAction) WHERE elementId(a) = $eid
            SET a.embedding = $vec
        """, {"eid": row['eid'], "vec": vec})

    # --- ClimateHazard ---
    hazards = kg.run("""
        MATCH (h:ClimateHazard)
        WHERE h.embedding IS NULL
        RETURN elementId(h) AS eid,
               coalesce(h.name, '') + ' ' +
               coalesce(h.hazard_type, '') AS text
    """)
    for row in tqdm(hazards, desc="Embedding ClimateHazard"):
        if not row['text'].strip():
            continue
        vec = embed(row['text'])
        kg.run_write("""
            MATCH (h:ClimateHazard) WHERE elementId(h) = $eid
            SET h.embedding = $vec
        """, {"eid": row['eid'], "vec": vec})

    # --- Policy ---
    policies = kg.run("""
        MATCH (p:Policy)
        WHERE p.embedding IS NULL
        RETURN elementId(p) AS eid,
               coalesce(p.policy_name, '') + ' ' +
               coalesce(p.policy_type, '') + ' ' +
               coalesce(p.level, '') AS text
    """)
    for row in tqdm(policies, desc="Embedding Policy"):
        if not row['text'].strip():
            continue
        vec = embed(row['text'])
        kg.run_write("""
            MATCH (p:Policy) WHERE elementId(p) = $eid
            SET p.embedding = $vec
        """, {"eid": row['eid'], "vec": vec})

    # --- Actor ---
    actors = kg.run("""
        MATCH (a:Actor)
        WHERE a.embedding IS NULL
        RETURN elementId(a) AS eid,
               coalesce(a.name, '') + ' ' +
               coalesce(a.sector, '') + ' ' +
               coalesce(a.role, '') AS text
    """)
    for row in tqdm(actors, desc="Embedding Actor"):
        if not row['text'].strip():
            continue
        vec = embed(row['text'])
        kg.run_write("""
            MATCH (a:Actor) WHERE elementId(a) = $eid
            SET a.embedding = $vec
        """, {"eid": row['eid'], "vec": vec})

    # --- Outcome ---
    outcomes = kg.run("""
        MATCH (o:Outcome)
        WHERE o.embedding IS NULL
        RETURN elementId(o) AS eid,
               coalesce(o.name, '') + ' ' +
               coalesce(o.outcome_type, '') AS text
    """)
    for row in tqdm(outcomes, desc="Embedding Outcome"):
        if not row['text'].strip():
            continue
        vec = embed(row['text'])
        kg.run_write("""
            MATCH (o:Outcome) WHERE elementId(o) = $eid
            SET o.embedding = $vec
        """, {"eid": row['eid'], "vec": vec})

    # vector
    index_configs = [
        ("node_embedding",    "AdaptationAction", "embedding"),
        ("hazard_embedding",  "ClimateHazard",    "embedding"),
        ("policy_embedding",  "Policy",           "embedding"),
        ("actor_embedding",   "Actor",            "embedding"),
        ("outcome_embedding", "Outcome",          "embedding"),
    ]
    for index_name, label, prop in index_configs:
        try:
            kg.run_write(f"""
                CREATE VECTOR INDEX {index_name} IF NOT EXISTS
                FOR (n:{label}) ON n.{prop}
                OPTIONS {{indexConfig: {{
                    `vector.dimensions`: 1536,
                    `vector.similarity_function`: 'cosine'
                }}}}
            """)
            print(f"  Index [{index_name}] created.")
        except Exception as e:
            print(f"  Index [{index_name}] already exists or error: {e}")

    print("All vector indexes ready.")

# just implement once
# build_vector_index()

# validation
for label, desc in [
    ("AdaptationAction", "actions"),
    ("ClimateHazard",    "hazards"),
    ("Policy",           "policies"),
    ("Actor",            "actors"),
    ("Outcome",          "outcomes"),
]:
    r = kg.run(f"MATCH (n:{label}) WHERE n.embedding IS NOT NULL RETURN count(n) AS cnt")
    print(f"{desc} with embedding: {r[0]['cnt']}")



# Vector Retrieval (Full Node Type)

def vector_retrieve(query: str, top_k: int = 5) -> dict:
    q_vec = embed(query)

    # AdaptationAction
    actions = kg.run("""
        CALL db.index.vector.queryNodes('node_embedding', $k, $vec)
        YIELD node, score
        MATCH (node)-[:LOCATED_IN]->(c:City)
        OPTIONAL MATCH (node)-[:ADDRESSES]->(h:ClimateHazard)
        OPTIONAL MATCH (node)-[:PRODUCES]->(o:Outcome)
        RETURN
            node.action_name AS name,
            'AdaptationAction' AS type,
            c.name AS city,
            collect(distinct h.name)[..3] AS hazards,
            collect(distinct o.name)[..3] AS outcomes,
            score
        ORDER BY score DESC
    """, {"k": top_k, "vec": q_vec})

    # ClimateHazard
    hazards = kg.run("""
        CALL db.index.vector.queryNodes('hazard_embedding', $k, $vec)
        YIELD node, score
        MATCH (c:City)-[:EXPERIENCES]->(node)
        OPTIONAL MATCH (a:AdaptationAction)-[:ADDRESSES]->(node)
        RETURN
            node.name AS name,
            'ClimateHazard' AS type,
            collect(distinct c.name)[..4] AS cities,
            collect(distinct a.action_name)[..3] AS actions,
            score
        ORDER BY score DESC
    """, {"k": top_k, "vec": q_vec})

    # Policy
    policies = kg.run("""
        CALL db.index.vector.queryNodes('policy_embedding', $k, $vec)
        YIELD node, score
        OPTIONAL MATCH (node)-[:MANDATES]->(a:AdaptationAction)-[:LOCATED_IN]->(c:City)
        OPTIONAL MATCH (node)-[:ISSUED_BY]->(actor:Actor)
        RETURN
            node.policy_name AS name,
            'Policy' AS type,
            node.policy_type AS policy_type,
            node.level AS level,
            collect(distinct c.name)[..3] AS cities,
            collect(distinct actor.name)[..2] AS issuers,
            score
        ORDER BY score DESC
    """, {"k": top_k, "vec": q_vec})

    # Actor
    actors = kg.run("""
        CALL db.index.vector.queryNodes('actor_embedding', $k, $vec)
        YIELD node, score
        OPTIONAL MATCH (node)-[:IMPLEMENTS]->(a:AdaptationAction)-[:LOCATED_IN]->(c:City)
        RETURN
            node.name AS name,
            'Actor' AS type,
            node.sector AS sector,
            node.role AS role,
            collect(distinct c.name)[..3] AS cities,
            collect(distinct a.action_name)[..3] AS actions,
            score
        ORDER BY score DESC
    """, {"k": top_k, "vec": q_vec})

    # Outcome
    outcomes = kg.run("""
        CALL db.index.vector.queryNodes('outcome_embedding', $k, $vec)
        YIELD node, score
        OPTIONAL MATCH (a:AdaptationAction)-[:PRODUCES]->(node)
        OPTIONAL MATCH (a)-[:LOCATED_IN]->(c:City)
        RETURN
            node.name AS name,
            'Outcome' AS type,
            node.outcome_type AS outcome_type,
            collect(distinct a.action_name)[..3] AS actions,
            collect(distinct c.name)[..3] AS cities,
            score
        ORDER BY score DESC
    """, {"k": top_k, "vec": q_vec})

    return {
        "actions":  actions,
        "hazards":  hazards,
        "policies": policies,
        "actors":   actors,
        "outcomes": outcomes,
    }

#Entity Linking

def entity_link(query: str) -> list:
    #LLM extraction
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{
            "role": "user",
            "content": f"""Extract entities from this query for a climate adaptation knowledge graph.
Return JSON only with keys: cities (list), hazards (list), actions (list), policies (list), actors (list).
Query: {query}"""
        }],
        response_format={"type": "json_object"},
        temperature=0.0
    )
    entities = json.loads(response.choices[0].message.content)
    candidates = []

    # City
    for city in entities.get("cities", []):
        result = kg.run("""
            MATCH (c:City)
            WHERE toLower(c.name) CONTAINS toLower($name)
            RETURN c.name AS name, 'City' AS type, elementId(c) AS eid
            LIMIT 3
        """, {"name": city})
        candidates.extend(result)

    # ClimateHazard
    for hazard in entities.get("hazards", []):
        result = kg.run("""
            MATCH (h:ClimateHazard)
            WHERE toLower(h.name) CONTAINS toLower($name)
            RETURN h.name AS name, 'ClimateHazard' AS type, elementId(h) AS eid
            LIMIT 3
        """, {"name": hazard})
        candidates.extend(result)

    # AdaptationAction
    for action in entities.get("actions", []):
        result = kg.run("""
            MATCH (a:AdaptationAction)
            WHERE toLower(a.action_name) CONTAINS toLower($name)
            RETURN a.action_name AS name, 'AdaptationAction' AS type, elementId(a) AS eid
            LIMIT 3
        """, {"name": action})
        candidates.extend(result)

    # Policy
    for policy in entities.get("policies", []):
        result = kg.run("""
            MATCH (p:Policy)
            WHERE toLower(p.policy_name) CONTAINS toLower($name)
            RETURN p.policy_name AS name, 'Policy' AS type, elementId(p) AS eid
            LIMIT 3
        """, {"name": policy})
        candidates.extend(result)

    # Actor
    for actor in entities.get("actors", []):
        result = kg.run("""
            MATCH (a:Actor)
            WHERE toLower(a.name) CONTAINS toLower($name)
            RETURN a.name AS name, 'Actor' AS type, elementId(a) AS eid
            LIMIT 3
        """, {"name": actor})
        candidates.extend(result)

    if not candidates:
        return []

    # llm filter
    candidate_labels = [f"{e['type']}: {e['name']}" for e in candidates]
    filter_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{
            "role": "user",
            "content": f"""Given this query: "{query}"

From the following candidate entities found in a climate adaptation knowledge graph,
select ONLY those directly relevant to answering the query.
Be strict — exclude entities that are only partially or coincidentally matching.

Candidates:
{json.dumps(candidate_labels, indent=2)}

Return JSON only with key: "selected" (list of strings, exactly as shown above)."""
        }],
        response_format={"type": "json_object"},
        temperature=0.0
    )

    selected = json.loads(
        filter_response.choices[0].message.content
    ).get("selected", [])

    linked = [
        e for e in candidates
        if f"{e['type']}: {e['name']}" in selected
    ]

    return linked



#multi-hop reasoning

def multihop_retrieve(linked_entities: list, max_hops: int = 2) -> dict:
    if not linked_entities:
        return {}

    subgraphs = {}

    for entity in linked_entities:
        eid   = entity['eid']
        name  = entity['name']
        etype = entity['type']

        if etype == 'City':
            causal = kg.run("""
                MATCH (c:City)-[:EXPERIENCES]->(h:ClimateHazard)
                      <-[:ADDRESSES]-(a:AdaptationAction)-[:PRODUCES]->(o:Outcome)
                WHERE elementId(c) = $eid
                RETURN c.name AS city,
                       h.name AS hazard,
                       a.action_name AS action,
                       collect(distinct o.name)[..3] AS outcomes,
                       a.cost_usd AS cost,
                       a.status AS status
                LIMIT 10
            """, {"eid": eid})

            governance = kg.run("""
                MATCH (p:Policy)-[:MANDATES]->(a:AdaptationAction)-[:LOCATED_IN]->(c:City)
                WHERE elementId(c) = $eid
                OPTIONAL MATCH (actor:Actor)-[:IMPLEMENTS]->(a)
                RETURN p.policy_name AS policy,
                       a.action_name AS action,
                       collect(distinct actor.name)[..3] AS implementors
                LIMIT 8
            """, {"eid": eid})

            subgraphs[name] = {"causal_chains": causal, "governance": governance}

        elif etype == 'ClimateHazard':
            result = kg.run("""
                MATCH (c:City)-[:EXPERIENCES]->(h:ClimateHazard)
                      <-[:ADDRESSES]-(a:AdaptationAction)
                WHERE elementId(h) = $eid
                OPTIONAL MATCH (a)-[:PRODUCES]->(o:Outcome)
                RETURN c.name AS city,
                       a.action_name AS action,
                       a.adaptation_type AS type,
                       collect(distinct o.name)[..2] AS outcomes
                LIMIT 15
            """, {"eid": eid})
            subgraphs[name] = {"city_responses": result}

        elif etype == 'AdaptationAction':
            result = kg.run("""
                MATCH (a:AdaptationAction)
                WHERE elementId(a) = $eid
                OPTIONAL MATCH (a)-[:LOCATED_IN]->(c:City)
                OPTIONAL MATCH (a)-[:ADDRESSES]->(h:ClimateHazard)
                OPTIONAL MATCH (a)-[:PRODUCES]->(o:Outcome)
                OPTIONAL MATCH (actor:Actor)-[:IMPLEMENTS]->(a)
                OPTIONAL MATCH (p:Policy)-[:MANDATES]->(a)
                OPTIONAL MATCH (a)-[:FACILITATED_BY]->(m:Mechanism)
                RETURN a.action_name AS action,
                       a.status AS status,
                       a.cost_usd AS cost,
                       c.name AS city,
                       collect(distinct h.name) AS hazards,
                       collect(distinct o.name) AS outcomes,
                       collect(distinct actor.name) AS implementors,
                       collect(distinct p.policy_name) AS policies,
                       collect(distinct m.name) AS mechanisms
            """, {"eid": eid})
            subgraphs[name] = {"action_detail": result}

        elif etype == 'Policy':
            result = kg.run("""
                MATCH (p:Policy)
                WHERE elementId(p) = $eid
                OPTIONAL MATCH (p)-[:MANDATES]->(a:AdaptationAction)-[:LOCATED_IN]->(c:City)
                OPTIONAL MATCH (p)-[:ISSUED_BY]->(actor:Actor)
                OPTIONAL MATCH (a)-[:ADDRESSES]->(h:ClimateHazard)
                RETURN p.policy_name AS policy,
                       p.policy_type AS policy_type,
                       p.level AS level,
                       collect(distinct c.name)[..4] AS cities,
                       collect(distinct a.action_name)[..5] AS actions,
                       collect(distinct actor.name)[..3] AS issuers,
                       collect(distinct h.name)[..3] AS hazards_addressed
                LIMIT 10
            """, {"eid": eid})
            subgraphs[name] = {"policy_detail": result}

        elif etype == 'Actor':
            result = kg.run("""
                MATCH (actor:Actor)
                WHERE elementId(actor) = $eid
                OPTIONAL MATCH (actor)-[:IMPLEMENTS]->(a:AdaptationAction)-[:LOCATED_IN]->(c:City)
                OPTIONAL MATCH (actor)-[:COORDINATES_WITH]->(other:Actor)
                RETURN actor.name AS actor,
                       actor.sector AS sector,
                       actor.role AS role,
                       collect(distinct c.name)[..4] AS cities,
                       collect(distinct a.action_name)[..5] AS actions,
                       collect(distinct other.name)[..3] AS collaborators
                LIMIT 10
            """, {"eid": eid})
            subgraphs[name] = {"actor_detail": result}

    return subgraphs


def subgraph_retrieve(linked_entities: list) -> list:
    if not linked_entities:
        return []

    eids = [e['eid'] for e in linked_entities]

    result = kg.run("""
        UNWIND $eids AS eid1
        UNWIND $eids AS eid2
        WITH eid1, eid2
        WHERE eid1 < eid2
        MATCH path = shortestPath((n1)-[*1..3]-(n2))
        WHERE elementId(n1) = eid1 AND elementId(n2) = eid2
        RETURN [node in nodes(path) |
                coalesce(node.name, node.action_name, node.policy_name)] AS node_names,
               [rel in relationships(path) | type(rel)] AS rel_types
        LIMIT 10
    """, {"eids": eids})

    return result



# LLM Cypher generation

CYPHER_SYSTEM = """You are a Neo4j Cypher expert for an urban climate adaptation knowledge graph.

Node properties (use ONLY these exact property names, no others):
- City:             name, region, country, population
- ClimateHazard:    name, hazard_type, severity
- AdaptationAction: action_name, adaptation_type, status, cost_usd, co_benefits
- Outcome:          name, outcome_type, magnitude
- Actor:            name, sector, role
- Policy:           policy_name, policy_type, level, year
- Mechanism:        name, mechanism_type
- Constraint:       name, constraint_type
- Vulnerability:    name, vulnerability_type
- UrbanZone:        name, zone_type
- Infrastructure:   name, infrastructure_type
- Indicator:        name, indicator_type, value
- ResilienceState:  name, state_type
- TimePoint:        name, year

Key relationships:
  (City)-[:EXPERIENCES]->(ClimateHazard)
  (ClimateHazard)<-[:ADDRESSES]-(AdaptationAction)
  (AdaptationAction)-[:PRODUCES]->(Outcome)
  (AdaptationAction)-[:LOCATED_IN]->(City)
  (Policy)-[:MANDATES]->(AdaptationAction)
  (Actor)-[:IMPLEMENTS]->(AdaptationAction)
  (Policy)-[:ISSUED_BY]->(Actor)
  (ClimateHazard)-[:WORSENS]->(Vulnerability)
  (Actor)-[:COORDINATES_WITH]->(Actor)
  (AdaptationAction)-[:FACILITATED_BY]->(Mechanism)
  (AdaptationAction)-[:HINDERED_BY]->(Constraint)

CRITICAL Rules:
1. Return ONLY valid Cypher. No explanation, no markdown, no backticks.
2. Always LIMIT results (max 20).
3. Use OPTIONAL MATCH for non-essential relationships.
4. Use toLower() for string matching.
5. Always return node properties, NOT node objects.
6. For ClimateHazard, always match by h.name (NOT h.type — type does not exist).
7. For AdaptationAction, always use a.action_name (NOT a.name).
8. For Policy, always use p.policy_name (NOT p.name).
9. For Actor, always use actor.name.
"""


def llm_cypher_generate(query: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": CYPHER_SYSTEM},
            {"role": "user",   "content": f"Generate a Cypher query for: {query}"}
        ],
        temperature=0.0
    )
    cypher = response.choices[0].message.content.strip()
    cypher = cypher.replace("```cypher", "").replace("```", "").strip()
    return cypher


def safe_cypher_execute(cypher: str) -> list:
    try:
        return kg.run(cypher)
    except Exception as e:
        print(f"    Cypher execution failed: {e}")
        return []



# Subgraph serialization

def serialize_context(
    vector_results: dict,
    graph_results: dict,
    cypher_results: list,
    subgraph_paths: list
) -> str:
    parts = []

    # Vector results
    if vector_results.get("actions"):
        parts.append("=== Relevant Adaptation Actions (semantic search) ===")
        for r in vector_results["actions"][:5]:
            parts.append(
                f"• [{r.get('city')}] {r.get('name')}"
                f"\n  Addresses: {', '.join(r.get('hazards', []))}"
                f"\n  Outcomes: {', '.join(r.get('outcomes', []))}"
                f"\n  Similarity: {r.get('score', 0):.3f}"
            )

    if vector_results.get("hazards"):
        parts.append("\n=== Related Climate Hazards (semantic search) ===")
        for r in vector_results["hazards"][:3]:
            parts.append(
                f"• {r.get('name')} — experienced by: {', '.join(r.get('cities', []))}"
                f"\n  Addressed by: {', '.join(r.get('actions', []))}"
            )

    if vector_results.get("policies"):
        parts.append("\n=== Relevant Policies (semantic search) ===")
        for r in vector_results["policies"][:3]:
            parts.append(
                f"• {r.get('name')} [{r.get('level')}]"
                f"\n  Cities: {', '.join(r.get('cities', []))}"
                f"\n  Issued by: {', '.join(r.get('issuers', []))}"
                f"\n  Similarity: {r.get('score', 0):.3f}"
            )

    if vector_results.get("actors"):
        parts.append("\n=== Relevant Actors (semantic search) ===")
        for r in vector_results["actors"][:3]:
            parts.append(
                f"• {r.get('name')} [{r.get('sector')}]"
                f"\n  Cities active in: {', '.join(r.get('cities', []))}"
                f"\n  Actions: {', '.join(r.get('actions', []))}"
                f"\n  Similarity: {r.get('score', 0):.3f}"
            )

    if vector_results.get("outcomes"):
        parts.append("\n=== Relevant Outcomes (semantic search) ===")
        for r in vector_results["outcomes"][:3]:
            parts.append(
                f"• {r.get('name')} [{r.get('outcome_type')}]"
                f"\n  Produced by: {', '.join(r.get('actions', []))}"
                f"\n  Cities: {', '.join(r.get('cities', []))}"
                f"\n  Similarity: {r.get('score', 0):.3f}"
            )

    # Graph traversal results
    if graph_results:
        parts.append("\n=== Graph Traversal Results ===")
        for entity_name, data in graph_results.items():
            parts.append(f"\nSubgraph around [{entity_name}]:")

            if "causal_chains" in data:
                for chain in data["causal_chains"][:5]:
                    parts.append(
                        f"  {chain.get('city')} → {chain.get('hazard')} "
                        f"→ {chain.get('action')} → {chain.get('outcomes', [])}"
                    )
            if "governance" in data:
                for g in data["governance"][:3]:
                    parts.append(
                        f"  Policy: {g.get('policy')} → {g.get('action')} "
                        f"(implemented by: {g.get('implementors', [])})"
                    )
            if "city_responses" in data:
                for r in data["city_responses"][:5]:
                    parts.append(
                        f"  {r.get('city')}: {r.get('action')} "
                        f"(type: {r.get('type')}) → {r.get('outcomes', [])}"
                    )
            if "action_detail" in data and data["action_detail"]:
                d = data["action_detail"][0]
                parts.append(
                    f"  Action: {d.get('action')}\n"
                    f"  City: {d.get('city')} | Status: {d.get('status')}\n"
                    f"  Hazards addressed: {d.get('hazards', [])}\n"
                    f"  Outcomes: {d.get('outcomes', [])}\n"
                    f"  Implemented by: {d.get('implementors', [])}\n"
                    f"  Policies: {d.get('policies', [])}"
                )
            if "policy_detail" in data and data["policy_detail"]:
                d = data["policy_detail"][0]
                parts.append(
                    f"  Policy: {d.get('policy')} [{d.get('policy_type')}]\n"
                    f"  Level: {d.get('level')}\n"
                    f"  Cities: {d.get('cities', [])}\n"
                    f"  Actions mandated: {d.get('actions', [])}\n"
                    f"  Issued by: {d.get('issuers', [])}"
                )
            if "actor_detail" in data and data["actor_detail"]:
                d = data["actor_detail"][0]
                parts.append(
                    f"  Actor: {d.get('actor')} [{d.get('sector')}]\n"
                    f"  Cities: {d.get('cities', [])}\n"
                    f"  Actions: {d.get('actions', [])}\n"
                    f"  Collaborators: {d.get('collaborators', [])}"
                )

    # Connecting paths
    if subgraph_paths:
        parts.append("\n=== Connecting Paths in Graph ===")
        for path in subgraph_paths[:5]:
            nodes = path.get("node_names", [])
            rels  = path.get("rel_types", [])
            path_str = ""
            for i, node in enumerate(nodes):
                path_str += str(node)
                if i < len(rels):
                    path_str += f" -[{rels[i]}]→ "
            parts.append(f"  {path_str}")

    # Cypher results
    if cypher_results:
        parts.append("\n=== Structured Query Results ===")
        for row in cypher_results[:8]:
            parts.append(f"  {dict(row)}")

    return "\n".join(parts)


# answer generation

ANSWER_SYSTEM = """You are an expert on urban climate adaptation policy.
Answer the user's question based ONLY on the provided knowledge graph context.

Rules:
1. Ground every claim in the retrieved context — do not hallucinate.
2. Cite specific cities, actions, or policies from the context.
3. If the context is insufficient, say so explicitly.
4. Structure your answer clearly with key findings first.
5. Keep answers concise (150-250 words).
"""


def generate_answer(query: str, context: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user",   "content":
             f"Context from knowledge graph:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0.2,
        max_tokens=500
    )
    return response.choices[0].message.content

# Reasoning Path Validity 

def validate_reasoning_paths(subgraph_paths: list) -> dict:
    """验证推理路径的well-formedness，计算ill-triplet rate"""
    if not subgraph_paths:
        return {"total_paths": 0, "well_formed": 0, "ill_formed": 0, "well_formed_rate": 1.0}

    total    = len(subgraph_paths)
    ill      = 0

    for path in subgraph_paths:
        nodes = path.get("node_names", [])
        rels  = path.get("rel_types", [])
        # Path disconnected: The number of nodes and the number of relationships do not match
        if len(nodes) != len(rels) + 1:
            ill += 1
            continue
        # None
        if any(n is None for n in nodes):
            ill += 1

    well_formed = total - ill
    return {
        "total_paths":     total,
        "well_formed":     well_formed,
        "ill_formed":      ill,
        "well_formed_rate": round(well_formed / total, 3) if total > 0 else 1.0,
    }


# QA Pipeline

def neo4j_json_serializer(obj):
    if hasattr(obj, 'isoformat'):
        return obj.isoformat()
    if hasattr(obj, '__str__'):
        return str(obj)
    raise TypeError(f'Object of type {obj.__class__.__name__} is not JSON serializable')


def qa_pipeline(query: str, verbose: bool = True) -> dict:
    if verbose:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print('='*60)

    if verbose: print("\n[1] Vector retrieval (5 node types)...")
    vector_results = vector_retrieve(query, top_k=5)

    if verbose: print("[2] Entity linking (with LLM filter)...")
    linked = entity_link(query)
    if verbose: print(f"    Linked: {[e['name'] for e in linked]}")

    if verbose: print("[3] Multi-hop graph traversal...")
    graph_results = multihop_retrieve(linked, max_hops=2)

    if verbose: print("[4] Subgraph path extraction...")
    subgraph_paths = subgraph_retrieve(linked)

    if verbose: print("[5] Validating reasoning paths...")
    path_validity = validate_reasoning_paths(subgraph_paths)
    if verbose: print(f"    Well-formed rate: {path_validity['well_formed_rate']:.3f} "
                      f"({path_validity['well_formed']}/{path_validity['total_paths']})")

    if verbose: print("[6] LLM Cypher generation...")
    cypher = llm_cypher_generate(query)
    if verbose: print(f"    Generated: {cypher[:100]}...")
    cypher_results = safe_cypher_execute(cypher)

    if verbose: print("[7] Serializing context...")
    context = serialize_context(
        vector_results, graph_results, cypher_results, subgraph_paths
    )

    if verbose: print("[8] Generating answer...")
    answer = generate_answer(query, context)

    if verbose:
        print(f"\n{'─'*60}")
        print("ANSWER:")
        print(answer)
        print(f"{'─'*60}")

    return {
        "query":            query,
        "linked_entities":  [dict(e) for e in linked],
        "cypher":           cypher,
        "cypher_results":   [dict(r) for r in cypher_results],
        "context":          context,
        "answer":           answer,
        "vector_hits": {
            k: len(v) for k, v in vector_results.items()
        },
        "graph_hits":       sum(len(v) for v in graph_results.values()),
        "path_validity":    path_validity,
    }


actions with embedding: 642
hazards with embedding: 205
policies with embedding: 106
actors with embedding: 119
outcomes with embedding: 317


In [13]:
graphrag_records = []

for item in queries:
    print(f"[{item['id']}/25] {item['question'][:60]}...")
    try:
        result = qa_pipeline(item['question'], verbose=False)
        graphrag_records.append({
            "id":       item['id'],
            "type":     item['type'],
            "question": item['question'],
            "answer":   result['answer'],
            "contexts": [result['context']],  
            "path_validity": result['path_validity']['well_formed_rate'],
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        graphrag_records.append({
            "id":       item['id'],
            "type":     item['type'],
            "question": item['question'],
            "answer":   "",
            "contexts": [""],
            "path_validity": 0.0,
        })
    time.sleep(1)

print(f"\nDone: {len(graphrag_records)} results collected")

with open('./data/graphrag_raw.json', 'w') as f:
    json.dump(graphrag_records, f, ensure_ascii=False, indent=2)

[1/25] What climate hazards does Houston face?...
[2/25] What adaptation actions has Vancouver implemented to address...
[3/25] What policies has Calgary issued for climate resilience?...
[4/25] What outcomes are produced by green stormwater infrastructur...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `magnitude` does not exist in database `kg200city`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=34, offset=209>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 209, 'line': 2, 'column': 34}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (c:City {name: 'Houston'})-[:EXPERIENCES]->(h:ClimateHazard)<-[:ADDRESSES]-(a:AdaptationAction {action_name: 'green stormwater infrastructure'})-[:PRODUCES]->(o:Outcome)\nRETURN o.name, o.outcome_type, o.magnitude\nLIMIT 20"


[5/25] Which actors are involved in climate governance in Washingto...
[6/25] What adaptation actions address extreme heat in Pittsburgh?...
[7/25] What vulnerabilities does Vancouver face related to sea leve...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `vulnerability_type` does not exist in database `kg200city`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=18, offset=159>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 159, 'line': 2, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (c:City {name: toLower("Vancouver")})-[:EXPERIENCES]->(h:ClimateHazard {name: toLower("sea level rise")})-[:WORSENS]->(v:Vulnerability)\nRETURN v.name, v.vulnerability_type\nLIMIT 20'


[8/25] What mechanisms does Calgary use to finance climate adaptati...
[9/25] Which cities have implemented nature-based solutions?...


...

[20/25] How do vulnerability assessments in Houston lead to specific...
[21/25] How do Vancouver and Calgary differ in their approaches to f...
[22/25] Which cities have the most diverse portfolio of adaptation a...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `country` does not exist in database `kg200city`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=28, offset=159>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 159, 'line': 4, 'column': 28}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (c:City)-[:LOCATED_IN]->(a:AdaptationAction)\nWITH c, COUNT(DISTINCT a.action_name) AS action_count\nORDER BY action_count DESC\nRETURN c.name, c.region, c.country, c.population, action_count\nLIMIT 20'


[23/25] Compare the governance structures of Houston and Pittsburgh ...
[24/25] Which cities address both flooding and extreme heat simultan...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `country` does not exist in database `kg200city`. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=28, offset=223>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 223, 'line': 5, 'column': 28}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (c:City)-[:EXPERIENCES]->(h:ClimateHazard)\nWHERE toLower(h.name) IN ['flooding', 'extreme heat']\nWITH c, COLLECT(h.name) AS hazards\nWHERE 'flooding' IN hazards AND 'extreme heat' IN hazards\nRETURN c.name, c.region, c.country, c.population\nLIMIT 20"


[25/25] How do Houston and Vancouver differ in their climate policy ...
    Cypher execution failed: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Variable `p` not defined (line 5, column 24 (offset: 253))
"RETURN c.name AS city, p.policy_name AS policy, m.name AS mechanism, o.name AS outcome, o.outcome_type AS outcome_type, o.magnitude AS outcome_magnitude"
                        ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

Done: 25 results collected


In [14]:
!pip install ragas langchain-openai datasets -q

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


/var/folders/r7/4ftfzdz96zs6t9xjbnmy5w3c0000gn/T/ipykernel_37368/3522955937.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/r7/4ftfzdz96zs6t9xjbnmy5w3c0000gn/T/ipykernel_37368/3522955937.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [15]:
langchain_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
langchain_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
ragas_llm = LangchainLLMWrapper(langchain_llm)
print("RAGAS ready")

RAGAS ready


/var/folders/r7/4ftfzdz96zs6t9xjbnmy5w3c0000gn/T/ipykernel_37368/3797308102.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(langchain_llm)


GraphRAG evaluation

In [16]:
# construct dataset from graphrag_records
valid_graphrag = [r for r in graphrag_records if r['answer']]

dataset_graphrag = Dataset.from_dict({
    "question": [r['question'] for r in valid_graphrag],
    "answer":   [r['answer']   for r in valid_graphrag],
    "contexts": [r['contexts'] for r in valid_graphrag],
})

print(f"Evaluating {len(valid_graphrag)} GraphRAG results...")
result_graphrag = evaluate(
    dataset_graphrag,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=langchain_embeddings,
)
df_graphrag = result_graphrag.to_pandas()
df_graphrag['system'] = 'GraphRAG'
df_graphrag['query_type'] = [r['type'] for r in valid_graphrag]
print(df_graphrag[['query_type','faithfulness','answer_relevancy']].to_string())

Evaluating 25 GraphRAG results...


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


    query_type  faithfulness  answer_relevancy
0   single_hop      1.000000          0.992838
1   single_hop      0.571429          0.961959
2   single_hop      0.666667          0.969910
3   single_hop      0.500000          0.961865
4   single_hop      0.600000          0.925722
5   single_hop      0.800000          0.880449
6   single_hop      0.461538          0.847438
7   single_hop      0.882353          0.000000
8   single_hop      0.750000          1.000000
9   single_hop      0.764706          0.903394
10   multi_hop      0.352941          0.881274
11   multi_hop      0.666667          0.998740
12   multi_hop      0.500000          0.728200
13   multi_hop      0.571429          0.791860
14   multi_hop      0.642857          0.939567
15   multi_hop      0.833333          0.707408
16   multi_hop      0.777778          0.000000
17   multi_hop      1.000000          0.845366
18   multi_hop      0.461538          0.686696
19   multi_hop      0.785714          0.000000
20  cross_cit

vector baseline

In [17]:
def vector_only_pipeline(query: str) -> dict:
    vector_results = vector_retrieve(query, top_k=5)
    parts = []
    for r in vector_results.get("actions", [])[:5]:
        parts.append(f"Action: {r.get('name')} in {r.get('city')}, "
                    f"addresses: {r.get('hazards')}, outcomes: {r.get('outcomes')}")
    for r in vector_results.get("hazards", [])[:3]:
        parts.append(f"Hazard: {r.get('name')} in cities: {r.get('cities')}")
    for r in vector_results.get("policies", [])[:3]:
        parts.append(f"Policy: {r.get('name')} in {r.get('cities')}")
    context = "\n".join(parts)
    answer = generate_answer(query, context)
    return {"answer": answer, "context": context}

import time
vector_records = []
for item in queries:
    print(f"[{item['id']}/25] {item['question'][:60]}...")
    try:
        result = vector_only_pipeline(item['question'])
        vector_records.append({
            "id": item['id'], "type": item['type'],
            "question": item['question'],
            "answer": result['answer'],
            "contexts": [result['context']],
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        vector_records.append({
            "id": item['id'], "type": item['type'],
            "question": item['question'],
            "answer": "", "contexts": [""],
        })
    time.sleep(0.5)

print(f"Done: {len(vector_records)} baseline results")

[1/25] What climate hazards does Houston face?...
[2/25] What adaptation actions has Vancouver implemented to address...
[3/25] What policies has Calgary issued for climate resilience?...
[4/25] What outcomes are produced by green stormwater infrastructur...
[5/25] Which actors are involved in climate governance in Washingto...
[6/25] What adaptation actions address extreme heat in Pittsburgh?...
[7/25] What vulnerabilities does Vancouver face related to sea leve...
[8/25] What mechanisms does Calgary use to finance climate adaptati...
[9/25] Which cities have implemented nature-based solutions?...
[10/25] What adaptation actions address flooding in Washington DC?...
[11/25] How do Houston's flood policies connect to specific measurab...
[12/25] What is the causal chain from extreme heat hazard to adaptat...
[13/25] Which actors mandate adaptation actions that produce resilie...
[14/25] How does flooding vulnerability in Pittsburgh lead to specif...
[15/25] What is the pathway from cli

summary

In [18]:
valid_vector = [r for r in vector_records if r['answer']]

dataset_vector = Dataset.from_dict({
    "question": [r['question'] for r in valid_vector],
    "answer":   [r['answer']   for r in valid_vector],
    "contexts": [r['contexts'] for r in valid_vector],
})

print(f"Evaluating {len(valid_vector)} Vector baseline results...")
result_vector = evaluate(
    dataset_vector,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=langchain_embeddings,
)
df_vector = result_vector.to_pandas()
df_vector['system'] = 'VectorRAG'
df_vector['query_type'] = [r['type'] for r in valid_vector]

df_all = pd.concat([df_graphrag, df_vector], ignore_index=True)
df_all.to_csv('./data/ragas_results.csv', index=False)

print("\n=== Overall Comparison ===")
print(df_all.groupby('system')[['faithfulness','answer_relevancy']].mean().round(3))

print("\n=== By Query Type ===")
print(df_all.groupby(['system','query_type'])[['faithfulness','answer_relevancy']].mean().round(3))

# Path validity
path_scores = [r['path_validity'] for r in graphrag_records if 'path_validity' in r]
print(f"\nGraphRAG Reasoning Path Well-formed Rate: {sum(path_scores)/len(path_scores):.3f}")

print("\nSaved → ./data/ragas_results.csv")

Evaluating 25 Vector baseline results...


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[36]: OutputParserException(Invalid json output: {
    "statements": [
        {
            "statement": "The provided context does not explicitly detail the relationship between Actor mandates and Outcome production in Pittsburgh.",
    


=== Overall Comparison ===
           faithfulness  answer_relevancy
system                                   
GraphRAG          0.699             0.740
VectorRAG         0.708             0.437

=== By Query Type ===
                      faithfulness  answer_relevancy
system    query_type                                
GraphRAG  cross_city         0.777             0.693
          multi_hop          0.659             0.658
          single_hop         0.700             0.844
VectorRAG cross_city         0.593             0.532
          multi_hop          0.731             0.253
          single_hop         0.744             0.574

GraphRAG Reasoning Path Well-formed Rate: 1.000

Saved → ./data/ragas_results.csv


In [20]:
summary = {
    "overall": df_all.groupby('system')[['faithfulness','answer_relevancy']].mean().round(3).to_dict(),
    "by_query_type": {
        f"{sys}_{qtype}": {
            "faithfulness": round(row['faithfulness'], 3),
            "answer_relevancy": round(row['answer_relevancy'], 3)
        }
        for (sys, qtype), row in df_all.groupby(['system','query_type'])[['faithfulness','answer_relevancy']].mean().iterrows()
    },
    "graphrag_path_validity": round(sum(path_scores)/len(path_scores), 3),
    "n_queries": len(queries),
    "key_finding": "GraphRAG outperforms VectorRAG in Answer Relevancy by +0.303 overall, with largest gap in multi-hop queries (+0.405)"
}

import json
with open('./data/evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("=== FINAL RESULTS ===")
print(f"GraphRAG Answer Relevancy: 0.740")
print(f"VectorRAG Answer Relevancy: 0.437")
print(f"Improvement: +0.303 (+69.3%)")
print(f"Multi-hop improvement: +0.405 (+160%)")
print(f"Path Well-formed Rate: 1.000")
print("\nSaved → ./data/evaluation_summary.json")

=== FINAL RESULTS ===
GraphRAG Answer Relevancy: 0.740
VectorRAG Answer Relevancy: 0.437
Improvement: +0.303 (+69.3%)
Multi-hop improvement: +0.405 (+160%)
Path Well-formed Rate: 1.000

Saved → ./data/evaluation_summary.json
